In [ ]:
import cv2
import mediapipe as mp
import numpy as np

mp_holistic = mp.solutions.holistic
holistic = mp_holistic.Holistic(
    static_image_mode=False, 
    model_complexity=1, # 1 is balanced for CPU-based real-time use
    min_detection_confidence=0.5, 
    min_tracking_confidence=0.5
)

I0000 00:00:1778376741.123224 27648478 gl_context.cc:369] GL version: 2.1 (2.1 Metal - 90.5), renderer: Apple M4 Pro


INFO: Created TensorFlow Lite XNNPACK delegate for CPU.


In [2]:
# Define these globally so the model knows the input size
FACE_INDICES = [33, 133, 362, 263, 61, 291, 13, 14, 105, 334] 

# Static points: Pose(33) + Face(len(FACE_INDICES)) + L_Hand(21) + R_Hand(21)
num_static_points = 33 + len(FACE_INDICES) + 21 + 21
# (x,y,z) * 2 (for Velocity)
input_features = (num_static_points * 3) * 2

def extract_landmarks(results):
    # Pose: 33 points
    if results.pose_landmarks:
        pose_coords = np.array([[res.x, res.y, res.z] for res in results.pose_landmarks.landmark])
        mid_shoulder = (pose_coords[11] + pose_coords[12]) / 2
        shoulder_dist = np.linalg.norm(pose_coords[11] - pose_coords[12])
        pose = ((pose_coords - mid_shoulder) / (shoulder_dist + 1e-6)).flatten()
    else:
        pose = np.zeros(33*3)

    # Hands: 21 points each
    def process_hand(hand_results):
        if not hand_results: return np.zeros(21*3)
        coords = np.array([[res.x, res.y, res.z] for res in hand_results.landmark])
        wrist = coords[0]
        hand_size = np.linalg.norm(coords[0] - coords[9])
        return ((coords - wrist) / (hand_size + 1e-6)).flatten()

    lh = process_hand(results.left_hand_landmarks)
    rh = process_hand(results.right_hand_landmarks)

    # Face: selected indices
    if results.face_landmarks:
        face_all = np.array([[res.x, res.y, res.z] for res in results.face_landmarks.landmark])
        face = (face_all[FACE_INDICES] - face_all[1]).flatten()
    else:
        face = np.zeros(len(FACE_INDICES)*3)

    return np.concatenate([pose, face, lh, rh])

def extract_refined_features(results, prev_landmarks=None):
    # Get current landmarks
    current_landmarks = extract_landmarks(results) 
    
    # If landmarks are all zeros (no detection), velocity must be zero
    if prev_landmarks is None or not np.any(current_landmarks):
        # Initial frame: velocity is zero
        velocity = np.zeros_like(current_landmarks)
    else:
        # Calculate Delta (Velocity) only if tracking is valid
        velocity = current_landmarks - prev_landmarks
    
    # Concatenate spatial position + velocity
    return np.concatenate([current_landmarks, velocity])

def build_sequence(video_path, max_frames=500):
    cap = cv2.VideoCapture(video_path)
    sequence = []
    prev_landmarks = None
    
    while cap.isOpened():
        ret, frame = cap.read()
        if not ret: break
        
        image = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        results = holistic.process(image)
        
        current_raw = extract_landmarks(results)
        feat = extract_refined_features(results, prev_landmarks)
        sequence.append(feat)
        
        if np.any(current_raw): 
            prev_landmarks = current_raw
            
        if len(sequence) >= max_frames: break
            
    cap.release()


In [ ]:
import pandas as pd
import os
from tqdm.notebook import tqdm

def process_how2sign(csv_path, video_dir, save_dir="how2sign_features"):
    if not os.path.exists(save_dir): os.makedirs(save_dir)
    
    df = pd.read_csv(csv_path, sep='\t')
    
    # We want to extract features for each video file.
    # But note: How2Sign videos are often full length, and sentences are crops.
    # We need to decide if we extract the whole video or just the sentence segment.
    # Since you have clips in the *_rgb_front_clips_raw_videos folders, we can process those!
    
    clips = [f for f in os.listdir(video_dir) if f.endswith('.mp4')]
    
    print(f"Found {len(clips)} videos to process in {video_dir}")
    for clip in tqdm(clips):
        video_path = os.path.join(video_dir, clip)
        vid_id = clip.replace('.mp4', '')
        save_path = os.path.join(save_dir, f"{vid_id}.npy")
        
        if os.path.exists(save_path):
            continue # Skip already processed
            
        # NOTE: For continuous ASL, we should NOT truncate to 30 frames.
        # We should extract the whole clip. You will need to modify build_sequence 
        # above to remove the 'sequence_length=30' and padding logic, and just extract all frames.
        features = build_sequence(video_path)
        np.save(save_path, features)

# Example usage (do not run yet until build_sequence is updated for continuous length):
process_how2sign('how2sign_realigned_test_glosses.csv', 'how2sign_dataset/test_rgb_front_clips_raw_videos', 'how2sign_features/test')
